# 00 - Ambiente

Confere se o projeto roda no computador local e no Google Colab: registro do ambiente,
predição com `yolo26n` e uma época de treino na amostra `coco8`.

No Colab, escolha antes **Ambiente de execução > Alterar o tipo > GPU**.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/Guilh3rme/epi-sistema.git"

if "COLAB_RELEASE_TAG" in os.environ:
    raiz = Path("/content/epi-sistema")
    if not raiz.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(raiz)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))  # dispensa reiniciar o ambiente
else:
    raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

(raiz / "runs").mkdir(exist_ok=True)
os.chdir(raiz / "runs")  # pesos e resultados ficam em runs/, fora do git
print("raiz:", raiz.name)

In [ ]:
from epi.ambiente import configurar_ultralytics, registrar

configurar_ultralytics(raiz)  # antes de qualquer importação do ultralytics
SEMENTE = 0
info = registrar(raiz / "runs" / "ambiente", semente=SEMENTE)
for chave, valor in info.items():
    print(f"{chave}: {valor}")
assert info["cuda_disponivel"], "Sem GPU: no Colab, altere o tipo do ambiente de execução." 

In [ ]:
from ultralytics import YOLO
from ultralytics.utils import ASSETS

modelo = YOLO("yolo26n.pt")
resultado = modelo.predict(ASSETS / "bus.jpg", device=0, verbose=False)[0]
classes = [resultado.names[int(c)] for c in resultado.boxes.cls]
print(classes)
assert "person" in classes

In [ ]:
treino = modelo.train(
    data="coco8.yaml", epochs=1, imgsz=640, batch=8, device=0,
    workers=0,  # evita processos filhos no Windows
    seed=SEMENTE, deterministic=True, plots=False,
    project=str(raiz / "runs"), name="fumaca", exist_ok=True,
)
assert (raiz / "runs" / "fumaca" / "weights" / "last.pt").exists()
print("ambiente OK")